# 3. MC context lambda

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances, roc_auc_score

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'
file_list = np.sort(glob(f'{indir}analysis/mC_context/L2any-donor/*.context.tsv'))


In [ ]:
group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
# group_meta = group_meta[['L2_any', 'L1', 'count']]
group_meta['L1_annot'] = group_meta['L1_annot'].str.replace(' ','-').str.replace('/','_')
annot2L1 = group_meta[['L1','L1_annot']].set_index('L1_annot')['L1'].to_dict()
L1annot = group_meta[['L1','L1_annot']].set_index('L1')['L1_annot'].to_dict()


In [ ]:
mc, cov, cluster = [], [], []
for file in file_list:
    tmp = pd.read_csv(file, index_col=0, header=None, names=['context', 'mc', 'cov'], sep='\t')
    mc.append(tmp['mc'])
    cov.append(tmp['cov'])
    cluster.append(file.split('/')[-1].split('.')[0])
    

In [ ]:
mc = pd.concat(mc, axis=1)
mc.columns = cluster
cov = pd.concat(cov, axis=1)
cov.columns = cluster


In [ ]:
selc = np.array([('N' not in xx) for xx in mc.index])
mc = mc.loc[selc].T
cov = cov.loc[selc].T


In [ ]:
mc.to_hdf('mC_context/L2any-donor_context.hdf', key='mc')
cov.to_hdf('mC_context/L2any-donor_context.hdf', key='cov')


In [ ]:
adata = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad')
adata

In [ ]:
mc_lambda = pd.read_csv('mC_context/cell_93350_lambda_context_mc.tsv.gz', sep='\t', header=0, index_col=0).loc[adata.obs.index]
cov_lambda = pd.read_csv('mC_context/cell_93350_lambda_context_cov.tsv.gz', sep='\t', header=0, index_col=0).loc[adata.obs.index]


In [ ]:
mc_lambda = mc_lambda.groupby(adata.obs['group']).sum().loc[mc.index]
cov_lambda = cov_lambda.groupby(adata.obs['group']).sum().loc[cov.index]

In [ ]:
mc_lambda.index = mc_lambda.index.astype(str)
cov_lambda.index = cov_lambda.index.astype(str)


In [ ]:
mc_lambda.to_hdf('mC_context/L2any-donor_lambda_context.hdf', key='mc')
cov_lambda.to_hdf('mC_context/L2any-donor_lambda_context.hdf', key='cov')


In [ ]:
mc = pd.read_hdf('mC_context/L2any-donor_context.hdf', key='mc')
cov = pd.read_hdf('mC_context/L2any-donor_context.hdf', key='cov')
mc_lambda = pd.read_hdf('mC_context/L2any-donor_lambda_context.hdf', key='mc')
cov_lambda = pd.read_hdf('mC_context/L2any-donor_lambda_context.hdf', key='cov')


In [ ]:
selch = np.array([(xx[:2]!='CG') for xx in mc.columns])


In [ ]:
tmp = np.log2(((mc+1)/(cov+1))/((mc_lambda+1)/(cov_lambda+1)))
tmp.index = ['-'.join([L1annot[xx.split('-')[0]]] + xx.split('-')[1:]) for xx in tmp.index]
sns.clustermap(tmp, cmap='coolwarm', vmax=1, vmin=-1, figsize=(10,20), yticklabels=8, metric='euclidean')

In [ ]:
groupL2 = pd.Series(['-'.join(xx.split('-')[:2]) for xx in mc.index], index=mc.index)
mc_L2 = mc.groupby(groupL2).sum()
cov_L2 = cov.groupby(groupL2).sum()
mc_lambda_L2 = mc_lambda.groupby(groupL2).sum()
cov_lambda_L2 = cov_lambda.groupby(groupL2).sum()

In [ ]:
tmp = np.log2(((mc_L2+1)/(cov_L2+1))/((mc_lambda_L2+1)/(cov_lambda_L2+1)))
tmp.index = ['-'.join([L1annot[xx.split('-')[0]]] + xx.split('-')[1:]) for xx in tmp.index]
sns.clustermap(tmp, cmap='coolwarm', vmax=2, vmin=-2, figsize=(10,25), yticklabels=3, metric='euclidean')

In [ ]:
groupL1 = pd.Series(mc.index.str.split('-').str[0], index=mc.index)
mc_L1 = mc.groupby(groupL1).sum().loc[:, selch]
cov_L1 = cov.groupby(groupL1).sum().loc[:, selch]
mc_lambda_L1 = mc_lambda.groupby(groupL1).sum().loc[:, selch]
cov_lambda_L1 = cov_lambda.groupby(groupL1).sum().loc[:, selch]


In [ ]:
tmp = np.log2(((mc_L1+1)/(cov_L1+1))/((mc_lambda_L1+1)/(cov_lambda_L1+1)))
tmp.index = ['-'.join([L1annot[xx.split('-')[0]]] + xx.split('-')[1:]) for xx in tmp.index]
sns.clustermap(tmp, cmap='coolwarm', vmax=2, vmin=-2, figsize=(10,10), yticklabels=1, metric='euclidean')


In [ ]:
tmp = (mc_L1+1)/(cov_L1+1)
tmp.index = ['-'.join([L1annot[xx.split('-')[0]]] + xx.split('-')[1:]) for xx in tmp.index]
sns.clustermap(tmp, cmap='coolwarm', vmax=0.05, vmin=0, figsize=(10,10), yticklabels=1, metric='cosine')


In [ ]:
tmp = (mc_lambda_L1+1)/(cov_lambda_L1+1)
tmp.index = ['-'.join([L1annot[xx.split('-')[0]]] + xx.split('-')[1:]) for xx in tmp.index]
sns.clustermap(tmp, cmap='coolwarm', vmax=0.05, vmin=0, figsize=(10,10), yticklabels=1, metric='cosine')


In [ ]:
colors = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)['color'].to_dict()


In [ ]:
tmp = (mc_L1+1)/(cov_L1+1)
tmp.index.name = 'L1'
tmp = tmp.stack().reset_index()
selc = tmp['L1'].isin(['c10', 'c16', 'c31'])
leg_order = tmp.groupby('context')[0].mean().sort_values().index[::-1]

fig, ax = plt.subplots(figsize=(4,2),dpi=300)
sns.stripplot(data=tmp, x='context', y=0, hue='L1', palette=colors, 
              order=leg_order, ax=ax, edgecolor='none', s=3)
ax.set_xticks(np.arange(selch.sum()))
ax.set_xticklabels(leg_order, rotation=90)
ax.set_ylabel('mC/C')
ax.get_legend().remove()
fig.tight_layout()


In [ ]:
# tmp = (mc_L1+1)/(cov_L1+1)
# tmp.index.name = 'L1'
# tmp = tmp.stack().reset_index()
# selc = tmp['L1'].isin(['c10', 'c16', 'c31'])
# leg_order = tmp.groupby('context')[0].mean().sort_values().index[::-1]

fig, ax = plt.subplots(figsize=(4,2.5),dpi=300)
sns.stripplot(data=tmp.loc[~selc], x='context', y=0, hue='L1', palette=colors, 
              order=leg_order, ax=ax, edgecolor='none', s=2)
ax.set_ylim([0, 0.045])
ax.set_yticks(np.arange(0, 0.05, 0.01))
ax.set_xticks(np.arange(selch.sum()))
ax.set_xticklabels(leg_order, rotation=90)
ax.set_ylabel('mC/C')
ax.get_legend().remove()
fig.tight_layout()
fig.savefig('mCH_distribution/L1_context_scatter.pdf', transparent=True)


In [ ]:
mc_ch_L2 = mc_L2.loc[:, selch]#.sum(axis=1)
cov_ch_L2 = cov_L2.loc[:, selch]#.sum(axis=1)
mc_lambda_ch_L2 = mc_lambda_L2.loc[:, selch]#.sum(axis=1)
cov_lambda_ch_L2 = cov_lambda_L2.loc[:, selch]#.sum(axis=1)


In [ ]:
tmp = np.log2(((mc_ch_L2+1)/(cov_ch_L2+1))/((mc_lambda_ch_L2+1)/(cov_lambda_ch_L2+1)))
tmp.index = ['-'.join([L1annot[xx.split('-')[0]]] + xx.split('-')[1:]) for xx in tmp.index]
sns.clustermap(tmp, cmap='coolwarm', vmax=2, vmin=-2, figsize=(10,25), yticklabels=3, metric='euclidean')

In [ ]:
L1_color = pd.read_csv(f'{indir}L1color.tsv', sep='\t', index_col=0, header=0)

In [ ]:
color = mc_ch_L2.index.str.split('-').str[0].map(L1_color['color'])
lambda_ch_L2 = mc_lambda_ch_L2/cov_lambda_ch_L2
ch_L2 = mc_ch_L2/cov_ch_L2

fig, axes = plt.subplots(2, 1, figsize=(4,8), dpi=300)
ax = axes[0]
ax.scatter(lambda_ch_L2, ch_L2, c=color, s=6, edgecolor='none')
ax = axes[1]
ax.scatter(lambda_ch_L2, ch_L2, c=color, s=6, edgecolor='none')
ax.set_xlim([0, 0.025])
ax.set_ylim([0, 0.025])
selc = ch_L2.index[(ch_L2-lambda_ch_L2>0.005) & (ch_L2<0.025)]
for xx in selc:
    ax.text(lambda_ch_L2.loc[xx], ch_L2.loc[xx], f'{L1annot[xx.split("-")[0]]}-{xx.split("-")[1]}', fontsize=6, ha='right', va='bottom')


In [ ]:
selc

In [ ]:
selcg = np.array([(xx[:2]=='CG') for xx in mc.columns])
selch = np.array([(xx[:2]!='CG') for xx in mc.columns])
mc_cg = mc.loc[:, selcg]
cov_cg = cov.loc[:, selcg]
mc_ch = mc.loc[:, selch]
cov_ch = cov.loc[:, selch]


In [ ]:
sns.clustermap((mc_cg / cov_cg).T, figsize=(20,4), xticklabels=4, cmap='Greens_r')

In [ ]:
sns.clustermap((mc / cov).corr(), figsize=(6,6), xticklabels=1, yticklabels=1, 
               metric='correlation', vmin=0.8, vmax=1, cmap='cividis')

In [ ]:
tmp = (mc_ch / cov_ch)
tmp = pd.DataFrame(tmp.max(axis=1).sort_values()[::-1])
tmp['L1'] = tmp.index.str.split('-').str[0].map(L1annot)
tmp = tmp.groupby('L1').head(2)

In [ ]:
selc = tmp.index[tmp[0]>0.025]
selc

In [ ]:
tmp.loc[selc]

In [ ]:
selc = pd.Index(['c12-c10', 'c13-c5', 'c13-c9', 'c14-c1', 'c14-c2', 'c14-c3', 'c14-c6',
       'c14-c7', 'c21-c8', 'c25-c10', 'c26-c10', 'c31-c0', 'c31-c1', 'c31-c2',
       'c31-c3', 'c31-c4', 'c31-c5', 'c31-c6', 'c31-c7', 'c32-c2', 'c5-c1',
       'c5-c2', 'c5-c5', 'c5-c9'])

In [ ]:
from ALLCools.mcds import MCDS

mcds = MCDS.open(f'{indir}merged_allc/cluster_donor.mcds', var_dim='chrom1k')
mcds

In [ ]:
mcds = mcds.assign_coords(L2_any=('cell', group_meta.loc[mcds.get_index('cell'), 'L2_any']))
mcds = mcds.groupby('L2_any').sum()
mcds = MCDS(mcds, obs_dim='L2_any', var_dim='chrom1k')

In [ ]:
mcds = mcds.sel({'mc_type':'CAN'})

In [ ]:
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='L2_any').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, log_scale=(True, False), ax=ax)
# ax.set_yscale('log')

In [ ]:
mcds = mcds.sel({'L2_any':selc, 'chrom1k':cov.index[cov>100]})

black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)

In [ ]:
# mcds.add_mc_frac(normalize_per_cell=False)


In [ ]:
# data = mcds['chrom1k_da_frac'].to_pandas()
mc = mcds.sel({'count_type':'mc'})['chrom1k_da'].to_pandas()
cov = mcds.sel({'count_type':'cov'})['chrom1k_da'].to_pandas()
data = mc / cov

In [ ]:
data.to_hdf('mC_context/L2any_mCA_025_top2_rawratio.hdf', key='data')


In [ ]:
from scipy.stats import betabinom, chi2
from scipy.optimize import minimize

def beta_binomial_log_likelihood(params, data):
    n, alpha, beta_param = params
    log_likelihood = np.sum(betabinom.logpmf(data, n, alpha, beta_param))
    return -log_likelihood


In [ ]:
data = pd.read_hdf('mC_context/L2any_mCA_025_top2_rawratio.hdf', key='data')


In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(12, 12))
for xx,ax in zip(data.index, axes.flatten()):
    tmp = data.loc[xx]
    tmp = tmp.loc[~tmp.isna()]
    cell_rate_mean = np.nanmean(tmp)
    cell_rate_var = np.nanvar(tmp)
    alpha = (1 - cell_rate_mean) * (cell_rate_mean**2) / cell_rate_var - cell_rate_mean
    beta = alpha * (1 / cell_rate_mean - 1)

    initial_params = [1, alpha, beta]
    result = minimize(beta_binomial_log_likelihood, initial_params, args=(tmp,))
    fitted_params = result.x
    n_fit, alpha_fit, beta_fit = fitted_params
    # print(initial_params, fitted_params)

    observed_freqs = np.histogram(tmp, bins=100)[0]
    expected_freqs = [betabinom.pmf(x, 99, alpha_fit, beta_fit) * len(tmp) for x in range(100)]
    observed = observed_freqs
    expected = np.array(expected_freqs)
    chi2_stat = np.sum(((observed - expected) ** 2) / expected)
    df = len(observed_freqs) - 1
    p_value = 1 - chi2.cdf(chi2_stat, df)
    print(xx, n_fit, alpha_fit, beta_fit, p_value)

    # sns.histplot(tmp, bins=100, ax=ax)
    ax.bar(np.arange(100), observed, width=1.0, color='C0')
    expected[expected<1] = 0
    ax.plot(np.arange(100), expected, color='C1')
    ax.set_title(f'{L1annot[xx.split("-")[0]]}-{xx.split("-")[1]}')
    # ax.set_yscale('log')
    
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(12, 12))
for xx,ax in zip(data.index, axes.flatten()):
    tmp = data.loc[xx]
    tmp = tmp.loc[~tmp.isna()]
    cell_rate_mean = np.nanmean(tmp)
    cell_rate_var = np.nanvar(tmp)
    alpha = (1 - cell_rate_mean) * (cell_rate_mean**2) / cell_rate_var - cell_rate_mean
    beta = alpha * (1 / cell_rate_mean - 1)

    initial_params = [1, alpha, beta]
    result = minimize(beta_binomial_log_likelihood, initial_params, args=(tmp,))
    fitted_params = result.x
    n_fit, alpha_fit, beta_fit = fitted_params
    # print(initial_params, fitted_params)

    observed_freqs = np.histogram(tmp, bins=100)[0]
    expected_freqs = [betabinom.pmf(x, 99, alpha_fit, beta_fit) * len(tmp) for x in range(100)]
    observed = observed_freqs
    expected = np.array(expected_freqs)
    chi2_stat = np.sum(((observed - expected) ** 2) / expected)
    df = len(observed_freqs) - 1
    p_value = 1 - chi2.cdf(chi2_stat, df)
    print(xx, n_fit, alpha_fit, beta_fit, p_value)

    # sns.histplot(tmp, bins=100, ax=ax)
    ax.bar(np.arange(100), observed, width=1.0, color='C0')
    expected[expected<1] = 0
    ax.plot(np.arange(100), expected, color='C1')
    ax.set_title(f'{L1annot[xx.split("-")[0]]}-{xx.split("-")[1]}')
    ax.set_yscale('log')
    
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(100), observed)
expected[expected<1] = 0
ax.plot(np.arange(100), expected)
ax.set_yscale('log')

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(100), observed)
expected[expected<1] = 0
ax.plot(np.arange(100), expected)
# ax.set_yscale('log')

In [ ]:
for xx in data.index:
    tmp = data.loc[xx]
    
    cell_rate_mean = np.nanmean(tmp)
    cell_rate_var = np.nanvar(tmp)
    alpha = (1 - cell_rate_mean) * (cell_rate_mean**2) / cell_rate_var - cell_rate_mean
    beta = alpha * (1 / cell_rate_mean - 1)
    
    initial_params = [1, alpha, beta]
    result = minimize(beta_binomial_log_likelihood, initial_params, args=(tmp,))
    fitted_params = result.x
    n_fit, alpha_fit, beta_fit = fitted_params
    
    # Create a DataFrame of observed frequencies
    observed_freqs = pd.Series(tmp).value_counts().sort_index()

    # Create a DataFrame of expected frequencies
    expected_freqs = [betabinom.pmf(x, n, alpha_fit, beta_fit) * len(data) for x in observed_freqs.index]

    # Chi-Square test
    observed = observed_freqs.values
    expected = np.array(expected_freqs)

    # Chi-square test statistic
    chi2_stat = np.sum(((observed - expected) ** 2) / expected)

    # Degrees of freedom
    df = len(observed_freqs) - 1

    # p-value
    p_value = 1 - chi2.cdf(chi2_stat, df)
    

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(12, 12))
for xx,ax in zip(data.index, axes.flatten()):
    tmp = data.loc[xx]
    sns.histplot(tmp, bins=100, ax=ax)
    ax.set_title(f'{L1annot[xx.split("-")[0]]}-{xx.split("-")[1]}')
    ax.set_yscale('log')
    
plt.tight_layout()


In [ ]:
data.to_hdf(f'{indir}merged_allc/L2any_mCGnorm.hdf', key='data')


In [ ]:
tmp = (mc_ch / cov_ch).loc[~mc_ch.index.str.split('-').str[0].isin(['c10','c16'])].T
tmp = tmp.loc[:, tmp.max(axis=0)>0.02]
tmp.columns = tmp.columns.str.split('-').str[0].map(L1annot) + '-' + tmp.columns.str.split('-').str[1]
sns.clustermap(tmp, figsize=(20,4), xticklabels=1, cmap='Purples')


In [ ]:
mc_ch_L1 = mc_ch.groupby(mc_ch.index.str.split('-').str[0]).sum()
cov_ch_L1 = cov_ch.groupby(cov_ch.index.str.split('-').str[0]).sum()


In [ ]:
group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
L1annot = group_meta[['L1','L1_annot']].drop_duplicates().set_index('L1')['L1_annot'].to_dict()


In [ ]:
L1annot

In [ ]:
tmp = (mc_ch_L1 / cov_ch_L1).T
tmp.columns = tmp.columns.map(L1annot)
corder = np.argsort(tmp.mean(axis=0))
sns.clustermap(tmp, figsize=(8,4), vmax=0.1, xticklabels=1, cmap='Purples')
